# Reel de la Tarea 3 · Trayectorias interrumpidas

Este cuaderno genera el reel con tu voz clonada (Qwen3-TTS), subtítulos con Whisper y música de fondo, usando el código de la sesión 15 del curso.

**Antes de empezar** ve a *Entorno de ejecución → Cambiar tipo de entorno de ejecución* y elige **GPU T4**. Luego ejecuta las celdas en orden con el botón ▶.

## Paso 1 · Descargar el código del curso y tu repositorio

In [ ]:
!git clone -q --depth 1 https://github.com/d2cml-ai/Data-Science-Python.git curso
!git clone -q --depth 1 https://github.com/anakaren-zl/data-science-python-2026.git mio
!ls mio/tarea_7/reel

## Paso 2 · Instalar las librerías (tarda unos 3 minutos)

In [ ]:
!pip install -q qwen-tts faster-whisper pillow imageio-ffmpeg soundfile jiwer num2words

## Paso 3 · Subir tu nota de voz

Sube la nota de voz de 15 a 25 segundos que grabaste (de WhatsApp sirve, en formato `.ogg`, `.opus` o `.m4a`).

In [ ]:
from google.colab import files
from pathlib import Path
subido = files.upload()
nombre = list(subido)[0]
voz = Path('mio/tarea_7/reel') / ('mi_voz' + Path(nombre).suffix)
voz.write_bytes(subido[nombre])
print('Voz guardada en', voz)

## Paso 4 · Revisar el diseño de cada escena (opcional, sin voz)

In [ ]:
!cd mio/tarea_7/reel && python hacer_reel.py --curso ../../../curso --solo-portadas
from IPython.display import Image, display
for i in range(1, 7):
    display(Image(f'mio/tarea_7/reel/prueba_escena_{i}.png', width=270))

## Paso 5 · Generar el reel (unos 5 minutos con GPU)

El script clona tu voz, narra cada escena, revisa con Whisper que se entienda (WER), crea los subtítulos, mezcla la música 22 dB por debajo de la voz y arma el video vertical de 1080 × 1920.

In [ ]:
!cd mio/tarea_7/reel && python hacer_reel.py --curso ../../../curso --voz $(ls mi_voz.*) --device cuda

## Paso 6 · Ver y descargar

## Paso 5b · Verificar la duración y el volumen

La rúbrica pide entre 45 y 75 segundos. Si queda fuera del rango, vuelve a ejecutar el paso 5 añadiendo `--velocidad 1.1` (más rápido) o `--velocidad 0.92` (más lento).

In [ ]:
import subprocess, json
d = float(subprocess.run(['ffprobe','-v','error','-show_entries','format=duration','-of','csv=p=0','mio/tarea_7/reel/reel.mp4'],capture_output=True,text=True).stdout)
print(f'Duración del reel {d:.1f} s', '✅' if 45 <= d <= 75 else '⚠️ fuera de 45 a 75 s')
g = json.load(open('mio/tarea_7/reel/guion.json', encoding='utf-8'))
print('Escenas', len(g['escenas']), '· WER por escena', [round(e.get('wer', 0), 2) for e in g['escenas']])

In [ ]:
from IPython.display import Video
display(Video('mio/tarea_7/reel/reel.mp4', embed=True, width=320))
for f in ['reel.mp4', 'subtitulos.srt', 'portada.png', 'guion.json']:
    files.download(f'mio/tarea_7/reel/{f}')

Sube `reel.mp4`, `subtitulos.srt`, `portada.png` y el `guion.json` actualizado (ahora trae el WER de cada escena) a la carpeta `tarea_7/reel/` de tu repositorio. **No subas tu nota de voz.**